#ปรับคุณภาพภาพ (Image Quality Enhancement)

**หัวข้อหลัก 2/4 ของ "ภาพรวม Image Processing"**

ภาพจากกล้องหน้างานจริงมักมืด/สว่างเกินไป หรือมี noise — บทนี้สอน **Brightness, Contrast, Gamma**
, **Histogram** , ลด **Noise** และ **CLAHE / Sharpen**
ด้วยภาพแผงวงจร (pcb) ภาพเดียวตลอดทั้งบท

> เนื้อหาและภาพตัวอย่างอ้างอิงจากสไลด์ **COMPUTER VISIONS** โดย Asst.Prof.Dr. Amnach Khawne, King Mongkut's Institute of Technology Ladkrabang

## วิธีใช้ Notebook นี้

- รันทีละ Cell จากบนลงล่างด้วย **Shift + Enter**
- แต่ละหัวข้อแบ่งเป็น Cell ย่อยหลาย Cell ทำทีละขั้นตอน เพื่อให้เห็นผลลัพธ์ทันทีทีละ Cell
- ลองแก้ค่าตัวเลข (parameter) แล้วรันซ้ำ เพื่อดูว่าภาพเปลี่ยนไปอย่างไร
- **ต้องอัปโหลด `data.zip` ก่อน** (Cell ที่ 2) ทุกครั้งที่เปิด Notebook ใหม่ — Colab ลบไฟล์ทิ้งเมื่อ Runtime ถูกรีเซ็ต


## 0. เตรียมเครื่องมือ (Setup)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow  # แสดงภาพใน Colab แทน cv2.imshow()

print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)


In [ ]:
def show_images(images, titles, cmap=None, figsize=(15, 5)):
    """แสดงภาพหลายภาพเรียงกันในแถวเดียว สำหรับเปรียบเทียบก่อน-หลัง"""
    n = len(images)
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(1, n, i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # BGR -> RGB สำหรับ matplotlib
        plt.title(title, fontsize=11)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 1. เตรียมชุดข้อมูล (Dataset) — อัปโหลด `data.zip`

อัปโหลดไฟล์ **`data.zip`** ที่ได้รับจากผู้สอน (ภาพชุดเดียวกับที่ใช้ในสไลด์ COMPUTER VISIONS)
เมื่อรัน Cell ด้านล่างจะมีปุ่มให้เลือกไฟล์จากเครื่อง — เลือก `data.zip` แล้วรอจนแตกไฟล์เสร็จ

In [ ]:
import os
import zipfile

from google.colab import files

%cd /content
print("เลือกไฟล์ data.zip (ชุดภาพตัวอย่างจากสไลด์ COMPUTER VISIONS)")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall("/content/")

DATA_DIR = "/content/data"
print(f"แตกไฟล์ {zip_filename} เรียบร้อยแล้ว")
print("ไฟล์ภาพที่มีในโฟลเดอร์ data/:")
for fname in sorted(os.listdir(DATA_DIR)):
    print(" -", fname)


## 2. โหลดภาพแผงวงจร (pcb.jpg)

In [ ]:
img = cv2.imread(f"{DATA_DIR}/pcb.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("shape:", img.shape, " ความสว่างเฉลี่ย:", round(gray.mean(), 1))


In [ ]:
cv2_imshow(img)


## 3. ปรับความสว่าง (Brightness) — `cv2.add()` / `cv2.subtract()`

parameter คือค่าคงที่ที่บวก/ลบทุกพิกเซล — ตรงกับสไลด์ 27 "Brightness +50"

In [ ]:
bright = cv2.add(img, np.full(img.shape, 50, dtype=np.uint8))
dark = cv2.subtract(img, np.full(img.shape, 50, dtype=np.uint8))


In [ ]:
show_images([dark, img, bright], ["-50", "Original", "+50"])
print("mean เดิม:", round(gray.mean(), 1),
      "| mean หลัง +50:", round(cv2.cvtColor(bright, cv2.COLOR_BGR2GRAY).mean(), 1))


## 4. ปรับ Contrast — `cv2.convertScaleAbs(img, alpha, beta)`

สูตร: `output = img * alpha + beta` — `alpha > 1` เพิ่ม contrast, `alpha < 1` ลด contrast

In [ ]:
low_contrast = cv2.convertScaleAbs(img, alpha=0.6, beta=0)
high_contrast = cv2.convertScaleAbs(img, alpha=1.6, beta=0)


In [ ]:
show_images([low_contrast, img, high_contrast], ["alpha=0.6", "Original", "alpha=1.6"])


## 5. ปรับ Gamma — `cv2.LUT()`

`output = 255 * (input/255) ** gamma` — `gamma < 1` ดึงรายละเอียดในเงามืดให้สว่างขึ้น, `gamma > 1` ทำให้มืดลง

In [ ]:
def build_gamma_table(gamma):
    return np.array([((i / 255.0) ** gamma) * 255 for i in range(256)]).astype("uint8")

gamma_bright = cv2.LUT(img, build_gamma_table(0.5))
gamma_dark = cv2.LUT(img, build_gamma_table(2.0))


In [ ]:
show_images([gamma_dark, img, gamma_bright], ["gamma=2.0 (darker)", "Original", "gamma=0.5 (brighter)"])


## 6. Histogram — `cv2.calcHist()`

แกน X = ระดับความเข้ม (0–255), แกน Y = จำนวนพิกเซล — บอกการกระจายของค่า แต่ไม่บอกตำแหน่งในภาพ

In [ ]:
hist_orig = cv2.calcHist([gray], [0], None, [256], [0, 256])
gray_bright = cv2.cvtColor(bright, cv2.COLOR_BGR2GRAY)
hist_bright = cv2.calcHist([gray_bright], [0], None, [256], [0, 256])


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(hist_orig, label="Original")
plt.plot(hist_bright, label="Brightness +50")
plt.xlabel("Intensity (0-255)")
plt.ylabel("Pixel count")
plt.legend()
plt.show()


## 7. ลด Noise: เลือก Filter ให้ตรงชนิด

- **Gaussian noise** (กระจายสุ่มทั่วภาพ) → `cv2.GaussianBlur()`
- **Salt & Pepper noise** (จุดขาว-ดำกระจัดกระจาย) → `cv2.medianBlur()`

In [ ]:
np.random.seed(42)
gauss_noise = np.random.normal(0, 25, img.shape).astype(np.float32)
noisy_gauss = np.clip(img.astype(np.float32) + gauss_noise, 0, 255).astype(np.uint8)
denoised_gauss = cv2.GaussianBlur(noisy_gauss, (5, 5), 0)

show_images([img, noisy_gauss, denoised_gauss], ["Original", "+ Gaussian noise", "GaussianBlur (5,5)"])


In [ ]:
noisy_sp = img.copy()
prob = 0.02
rnd = np.random.rand(*img.shape[:2])
noisy_sp[rnd < prob / 2] = 0
noisy_sp[rnd > 1 - prob / 2] = 255
denoised_median = cv2.medianBlur(noisy_sp, 3)

show_images([img, noisy_sp, denoised_median], ["Original", "+ Salt & Pepper", "medianBlur (3)"])


## 8. เพิ่ม Contrast เฉพาะจุด — CLAHE vs `equalizeHist()`

- `cv2.equalizeHist()` ปรับทั้งภาพ (Global)
- `cv2.createCLAHE(clipLimit, tileGridSize)` ปรับทีละบริเวณย่อย (Local/Adaptive)

In [ ]:
equalized = cv2.equalizeHist(gray)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_result = clahe.apply(gray)


In [ ]:
show_images([gray, equalized, clahe_result], ["Original gray", "equalizeHist", "CLAHE"], cmap="gray")
print("std เดิม:", round(gray.std(), 1), "| หลัง CLAHE:", round(clahe_result.std(), 1))


## 9. เน้นขอบภาพ (Sharpen) — Unsharp Mask

เบลอภาพก่อน แล้วเอาต้นฉบับลบภาพเบลอออกบางส่วนด้วย `cv2.addWeighted()` เพื่อขยายความต่างบริเวณขอบ

In [ ]:
blur_for_sharpen = cv2.GaussianBlur(img, (0, 0), 3)
sharpened = cv2.addWeighted(img, 1.5, blur_for_sharpen, -0.5, 0)


In [ ]:
show_images([img, blur_for_sharpen, sharpened], ["Original", "Blurred", "Sharpened"])


## สรุปสิ่งที่เรียนในบทนี้

| แนวคิด | คำสั่งที่ใช้ | Parameter สำคัญ |
|---|---|---|
| Brightness | `cv2.add()` / `cv2.subtract()` | ค่าคงที่ที่บวก/ลบ |
| Contrast | `cv2.convertScaleAbs(alpha, beta)` | alpha, beta |
| Gamma | `cv2.LUT()` | gamma |
| Histogram | `cv2.calcHist()` | - |
| ลด Gaussian noise | `cv2.GaussianBlur(ksize)` | ksize |
| ลด Salt&Pepper noise | `cv2.medianBlur(ksize)` | ksize |
| Contrast เฉพาะจุด | `cv2.createCLAHE(clipLimit, tileGridSize)` | clipLimit, tileGridSize |
| Sharpen | `cv2.GaussianBlur` + `cv2.addWeighted` | sigma, น้ำหนัก |

**บทถัดไป:** แยกบริเวณที่สนใจ (Threshold / HSV / Mask / Morphology)


## แบบฝึกหัดท้ายบท (ลองทำเอง)

1. เปลี่ยนค่า `alpha` ในหัวข้อ 4 เป็น 2.0 หรือ 0.3
2. เปลี่ยน `gamma` ในหัวข้อ 5 เป็น 0.3 และ 3.0
3. เปลี่ยน `clipLimit` ของ CLAHE ในหัวข้อ 8 เป็น 4.0
